# H3: level offsets of ODYSSEY periods A and B, checked with CERRA-Land

H3 (#20), second pull request (comment of 13/11/2026 on #79; H3b, #82, and H4, #23, merged into H3). ODYSSEY is
three products in a row (`docs/method.md`, *Radars over time*): A 4/2013–11/2015, B 12/2015–7/2018, C 8/2018–6/2024;
D is the target, CIRRUS 7/2024–17/11/2025. Period C is in notebook 07, section 12; here the same procedure for A
and B, with ERA5-Land as the bridge, and CERRA-Land as the independent check of ERA5-Land at 12/2015 and 8/2018.

1. **Data:** per-cell monthly sums of radar and ERA5-Land, as notebook 07 section 12, from 4/2013; Belgian-area
   daily sums on 06–06 UTC days of radar, ERA5-Land and CERRA-Land.
2. **Offsets of A, B (and C) against D** with ERA5-Land.
3. **CERRA-Land check** of the steps at 12/2015 and 8/2018, and offsets with CERRA-Land as bridge.
4. **Seasons, rings of distance from the radar, continuity** at 12/2015, 8/2018 and 7/2024 with years left out.
5. **Figure.**
6. **Uncertainty of the level, and residual of one offset.**
7. **Volume or intensity:** the offsets split into how often and how hard it rains.

Results in `docs/method.md`, *Level offsets of periods A and B (second H3 pull request)*.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import calendar
import datetime as dt
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy import stats
from scipy.stats import f_oneway, mannwhitneyu, ttest_ind

OUT = RESULTS_DIR / "h3_level"                      # hourly sums on the ERA5-Land grid, notebook 07 sections 4 and 9
FIG = Path.cwd().parent / "docs" / "figures" / "h3-level"
FULL_FRAMES = {"odyssey": 4, "cirrus": 12}
D_END = np.datetime64("2025-11-18T00")             # target D: CIRRUS to 17/11/2025
PERIODS = {"A": ((2013, 4), (2015, 11)), "B": ((2015, 12), (2018, 7)), "C": ((2018, 8), (2024, 6)),
           "D": ((2024, 7), (2025, 11))}

era = xr.open_zarr(DATA_DIR / "era5land_bel.zarr", consolidated=False)
expver = era["expver"].load()


def full_cells(product):
    z = np.load(OUT / f"era5grid_{product}_2024-07.npz")
    return {tuple(c) for c in z["cells"][z["npix"] >= 0.8 * z["npix"].max()]}


COMMON = sorted(full_cells("odyssey") & full_cells("cirrus"))     # the 186 cells of notebook 07
CI, CJ = np.array([c[0] for c in COMMON]), np.array([c[1] for c in COMMON])
print(len(COMMON), "cells")

## 1. Data

- **Monthly, per cell:** complete radar hours and final ERA5-Land, ODYSSEY cleaned as in H2 (notebook 07, section
  9), CIRRUS as it is; months with less than half of their hours are left out (11/2025: of its 17 days).
- **Daily, 06–06 UTC, Belgian area:** the hours where all 186 cells are finite in both are averaged over the
  cells; the day labelled *d* 06 UTC takes the hours ending in (*d*−1 06, *d* 06], as CERRA-Land (notebook 08);
  only days with all 24 hours. CERRA-Land is averaged over its cells whose centre lies in the 0.1° box of one of
  the 186 cells. So ODYSSEY is not recomputed on the CERRA-Land grid: the check uses the same area.

In [ ]:
MONTHS, H, RM, EM = {}, [], [], []
for y in range(2013, 2026):
    for m in range(1, 13):
        if not (2013, 4) <= (y, m) <= (2025, 11):
            continue
        product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
        ym = f"{y}-{m:02d}"
        z = np.load(OUT / (f"era5grid_odyssey_clean_{ym}.npz" if product == "odyssey" else f"era5grid_cirrus_{ym}.npz"))
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        sel = np.array([pos[c] for c in COMMON])
        he = np.array(z["hour_end"], dtype="datetime64[ns]")
        ok = (z["frames"] == FULL_FRAMES[product]) & (he <= era["valid_time"].values[-1]) & (he <= D_END)
        final = expver.sel(valid_time=he[ok]).values == 1
        hours = he[ok][final]
        r = (z["rain_clean"] if product == "odyssey" else z["rain"])[ok][final][:, sel]
        e = era["tp"].sel(valid_time=hours).values[:, CI, CJ]
        good = np.isfinite(r) & np.isfinite(e)
        H.append(hours[good.all(1)]); RM.append(r[good.all(1)].mean(1)); EM.append(e[good.all(1)].mean(1))
        nmax = (17 if ym == "2025-11" else calendar.monthrange(y, m)[1]) * 24
        if hours.size >= 0.5 * nmax and final.mean() >= 0.5:
            MONTHS[(y, m)] = dict(radar=np.where(good, r, 0).sum(0), era5=np.where(good, e, 0).sum(0))
KEYS = {p: [k for k in MONTHS if a <= k <= b] for p, (a, b) in PERIODS.items()}

H, RM, EM = np.concatenate(H).astype("datetime64[h]"), np.concatenate(RM), np.concatenate(EM)
lab = (H - np.timedelta64(7, "h")).astype("datetime64[D]").astype("datetime64[h]") + np.timedelta64(30, "h")
days, inv, cnt = np.unique(lab, return_inverse=True, return_counts=True)
rd, ed = np.bincount(inv, RM)[cnt == 24], np.bincount(inv, EM)[cnt == 24]
days = days[cnt == 24]
cer = xr.open_zarr(DATA_DIR / "cerraland_bel.zarr", consolidated=False)
inside = np.zeros(cer.latitude.shape, bool)
for la, lo in zip(era["latitude"].values[CI], era["longitude"].values[CJ]):
    inside |= (np.abs(cer.latitude.values - la) <= 0.05) & (np.abs(cer.longitude.values - lo) <= 0.05)
cv = cer.valid_time.values.astype("datetime64[h]")
keep = np.isin(days, cv)
days, rd, ed = days[keep], rd[keep], ed[keep]
cd = np.nanmean(cer["tp"].values[np.isin(cv, days)][:, inside], 1)
print(f"{len(MONTHS)} months {min(MONTHS)} → {max(MONTHS)}; {', '.join(f'{p} {len(v)}' for p, v in KEYS.items())}")
print(f"{days.size} complete 06–06 days; {inside.sum()} CERRA-Land cells; day check, correlation ERA5-Land /"
      f" CERRA-Land same day {np.corrcoef(ed, cd)[0, 1]:.3f}, one day apart {np.corrcoef(ed[1:], cd[:-1])[0, 1]:.3f}")

## 2. Offsets of A, B and C against D, ERA5-Land

As notebook 07, section 12: log10 of radar / ERA5-Land per month; for each calendar month, the mean over D minus
the mean over the period; the offset is the mean of the 12 differences, with a 95% interval (t, 11 degrees of
freedom); rain × k is +16·log10 k dB. Checks without the artefact months of B (spring 2018) and the summer echoes
near the radars of A (summers 2014 and 2015; `docs/method.md`, *Radars over time*).

In [ ]:
def logr(keys, cells=slice(None)):
    return np.array([np.log10(MONTHS[k]["radar"][cells].sum() / MONTHS[k]["era5"][cells].sum()) for k in keys])


def offset(c_keys, d_keys, **kw):
    lc, ld = logr(c_keys, **kw), logr(d_keys, **kw)
    mc, md = np.array([k[1] for k in c_keys]), np.array([k[1] for k in d_keys])
    d = np.array([ld[md == m].mean() - lc[mc == m].mean() for m in range(1, 13)])
    half = stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12)
    return dict(off=d.mean(), lo=d.mean() - half, hi=d.mean() + half, by_month=d, unbalanced=ld.mean() - lc.mean())


def show(name, o):
    print(f"{name:46s} {o['off']:+.3f} (×{10 ** o['off']:.2f}, {16 * o['off']:+.2f} dB), 95% ×{10 ** o['lo']:.2f}–"
          f"{10 ** o['hi']:.2f} ({16 * o['lo']:+.2f} to {16 * o['hi']:+.2f} dB); unbalanced {o['unbalanced']:+.3f}")


OFF = {p: offset(KEYS[p], KEYS["D"]) for p in "ABC"}
for p in "ABC":
    show(f"{p} against D", OFF[p])
show("B without 3–6/2018", offset([k for k in KEYS["B"] if not (2018, 3) <= k <= (2018, 6)], KEYS["D"]))
show("A without the summers 2014 and 2015", offset([k for k in KEYS["A"] if not (k[1] in (6, 7, 8) and k[0] in (2014, 2015))], KEYS["D"]))

## 3. CERRA-Land: the steps at 12/2015 and 8/2018

Monthly sums of the complete 06–06 days (all 152 months have at least 15). The step of `opera_ds.homog.step`
(12 and 24 months each side, t-test and Wilcoxon–Mann–Whitney) on radar / ERA5-Land, radar / CERRA-Land and
ERA5-Land / CERRA-Land: if the radar steps are the same with both bridges and the ratio of the bridges has no
step, the step is in the radar. Then the offsets of A, B and C against D with each bridge on the same days; the
rule of the comment of 13/11 on #79 (VMM against ERA5-Land): if the intervals overlap, ERA5-Land carries the
period, else the difference is an uncertainty to state at G1.

In [ ]:
from opera_ds.homog import step

ym = days.astype("datetime64[M]")
MON = {}
for m in np.unique(ym):
    k = ym == m
    if k.sum() >= 15:
        MON[(int(str(m)[:4]), int(str(m)[5:7]))] = dict(radar=rd[k].sum(), era5=ed[k].sum(), cerra=cd[k].sum())
print(len(MON), "months with at least 15 complete days")
SER = {"radar / ERA5-Land": ("radar", "era5"), "radar / CERRA-Land": ("radar", "cerra"),
       "ERA5-Land / CERRA-Land": ("era5", "cerra")}
for at in ((2015, 12), (2018, 8), (2024, 7)):
    for n in (12, 24):
        print(f"step at {at[1]}/{at[0]}, {n} months each side")
        for name, (a, b) in SER.items():
            s = step({k: v[a] / v[b] for k, v in MON.items()}, at, n)
            print(f"   {name:24s} {s['step']:+.3f} (×{10 ** s['step']:.2f}), t-test p {s['t_p']:.3f},"
                  f" Mann–Whitney p {s['mw_p']:.3f}, n {s['n_before']}/{s['n_after']}")


def offset_daily(p, den):
    lr = {k: np.log10(v["radar"] / v[den]) for k, v in MON.items()}
    a, b = PERIODS[p]
    d = np.array([np.mean([v for k, v in lr.items() if k[1] == m and PERIODS["D"][0] <= k]) -
                  np.mean([v for k, v in lr.items() if k[1] == m and a <= k <= b]) for m in range(1, 13)])
    half = stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12)
    return d.mean(), d.mean() - half, d.mean() + half


print("\noffsets against D on the same 06–06 days")
for p in "ABC":
    (e, elo, ehi), (c, clo, chi) = offset_daily(p, "era5"), offset_daily(p, "cerra")
    print(f"   {p}: ERA5-Land {e:+.3f} (×{10 ** e:.2f}, ×{10 ** elo:.2f}–{10 ** ehi:.2f}), CERRA-Land {c:+.3f}"
          f" (×{10 ** c:.2f}, ×{10 ** clo:.2f}–{10 ** chi:.2f}); difference {c - e:+.3f};"
          f" intervals overlap: {max(elo, clo) <= min(ehi, chi)}")

## 4. Seasons, rings, continuity

- **Seasons:** as notebook 07, section 12; A and B have only 2–3 years of each season.
- **Rings:** distance of each cell from the nearest of the radars of notebook 07, section 11; the offset of each
  ring, and the local 12-month step of radar / ERA5-Land at 12/2015 and 8/2018 in each ring.
- **Continuity:** each period split into 12-month blocks aligned to its end (the first block may be shorter);
  each block is left out in turn, the offset refitted on the rest of its period and applied to it. The yardstick
  is the standard deviation of the corrected means of the full blocks of A, B and C, with its 95% interval
  (chi-square). The jump at a boundary is the corrected first 12 months after it (offset fitted without them;
  for D, CIRRUS as it is) minus the corrected last 12 months before it (offset fitted without them).

In [ ]:
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}
for p in "AB":
    groups = [OFF[p]["by_month"][[m - 1 for m in ms]] for ms in SEASONS.values()]
    print(f"{p}: one-way ANOVA of the 12 monthly differences by season, p {f_oneway(*groups).pvalue:.2f}")
    for (s, ms), g in zip(SEASONS.items(), groups):
        ck = [k for k in KEYS[p] if k[1] in ms]
        lc, sy = logr(ck), np.array([k[0] + (k[1] == 12) for k in ck])
        yrs = [lc[sy == y].mean() for y in np.unique(sy) if (sy == y).sum() == 3]
        sd = np.std(yrs, ddof=1)
        dev = g.mean() - OFF[p]["off"]
        print(f"   {s}: offset {g.mean():+.3f} (×{10 ** g.mean():.2f}), minus the single offset {dev:+.3f};"
              f" year-to-year sd {sd:.3f} ({len(yrs)} years) → {'exceeds' if abs(dev) > sd else 'within'}")

RADARS_NEAR = ["bejab", "behel", "bewid", "bezav", "nldhl", "nlhrw", "frabb", "frave", "deess"]
DB = {r["odimcode"]: (float(r["longitude"]), float(r["latitude"]))
      for r in json.load(open(DATA_DIR / "opera_radars" / "OPERA_RADARS_DB_20261110.json", encoding="utf-8"))
      if r.get("odimcode") in RADARS_NEAR}


def km(lon1, lat1, lon2, lat2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


DIST = np.min([km(era["longitude"].values[CJ], era["latitude"].values[CI], lo, la) for lo, la in DB.values()], axis=0)
print(f"\nrings, cells {DIST.min():.0f}–{DIST.max():.0f} km from the nearest radar")
for name, a, b in (("0–50 km", 0, 50), ("50–100 km", 50, 100)):
    cells = np.where((DIST >= a) & (DIST < b))[0]
    for p in "ABC":
        show(f"{p}, {name}, {cells.size} cells", offset(KEYS[p], KEYS["D"], cells=cells))
    for at_name, before, after in (("12/2015", KEYS["A"][-12:], KEYS["B"][:12]), ("8/2018", KEYS["B"][-12:], KEYS["C"][:12])):
        print(f"   {name}: local 12-month step at {at_name} {logr(after, cells).mean() - logr(before, cells).mean():+.3f}")

In [ ]:
def blocks(p):
    ks, out = KEYS[p], []
    while ks:
        out.insert(0, ks[-12:])
        ks = ks[:-12]
    return out


def corrected(p, blk, exclude=()):
    o = offset([k for k in KEYS[p] if k not in blk and k not in exclude], KEYS["D"])["off"]
    blk = [k for k in blk if k not in exclude]
    return (logr(blk) + o).mean(), o


ROWS = []
for p in "ABC":
    for blk in blocks(p):
        m, o = corrected(p, blk)
        ROWS.append((p, blk, m))
        print(f"{p} {blk[0][1]}/{blk[0][0]}–{blk[-1][1]}/{blk[-1][0]} ({len(blk)} months): offset of the rest {o:+.3f},"
              f" corrected mean {m:+.3f}")
full = np.array([m for p, blk, m in ROWS if len(blk) == 12])
n, YARD = full.size, full.std(ddof=1)
Y_LO, Y_HI = (YARD * np.sqrt((n - 1) / stats.chi2.ppf(q, n - 1)) for q in (0.975, 0.025))
print(f"yardstick: {YARD:.3f}, 95% {Y_LO:.3f}–{Y_HI:.3f} ({n} blocks)")
D12 = logr([k for k in KEYS["D"] if k <= (2025, 6)]).mean()
JUMPS = {}
for p, q in (("A", "B"), ("B", "C"), ("C", "D")):
    before = corrected(p, KEYS[p][-12:])[0]
    after = D12 if q == "D" else corrected(q, KEYS[q][:12])[0]
    JUMPS[(p, q)] = (before, after)
    print(f"jump {p}→{q}: {after - before:+.3f} (×{10 ** (after - before):.2f}) → {'within' if abs(after - before) <= YARD else 'larger than'} the yardstick")
spring18 = [(2018, m) for m in (3, 4, 5, 6)]
b8 = corrected("B", KEYS["B"][-12:], exclude=spring18)[0]
print(f"jump B→C without 3–6/2018 in the fit and in the block: {JUMPS[('B', 'C')][1] - b8:+.3f}")

## 5. Figure

In [ ]:
keys = [k for p in "ABCD" for k in KEYS[p]]
t = np.array([dt.datetime(y, m, 15) for y, m in keys])
lr = logr(keys)
per = np.array([next(p for p in "ABCD" if PERIODS[p][0] <= k <= PERIODS[p][1]) for k in keys])
corr = lr + np.array([OFF[p]["off"] if p in OFF else 0.0 for p in per])
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False, constrained_layout=True, height_ratios=[2, 1])
ax = axes[0]
ax.plot(t[per != "D"], 10 ** lr[per != "D"], ".-", color="0.7", lw=0.7, label="ODYSSEY cleaned")
for p, colour in (("A", "tab:green"), ("B", "tab:orange"), ("C", "tab:blue")):
    k = per == p
    ax.plot(t[k], 10 ** corr[k], ".-", color=colour, lw=0.9, label=f"{p} × {10 ** OFF[p]['off']:.2f}")
ax.plot(t[per == "D"], 10 ** lr[per == "D"], ".-", color="tab:red", lw=0.9, label="CIRRUS, target D")
for p, blk, m in ROWS:
    ax.hlines(10 ** m, dt.datetime(*blk[0], 1), dt.datetime(*blk[-1], 28), color="k", lw=2, alpha=0.35)
ax.set_yscale("log")
ax.set_ylim(1.0, 11)
ax.set_yticks([1, 1.5, 2, 3, 4, 6, 8, 10])
ax.set_yticklabels(["1", "1.5", "2", "3", "4", "6", "8", "10"])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_ylabel("radar / ERA5-Land\n(monthly totals)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=5, fontsize=8.5)
ax.set_title("Belgium, 186 ERA5-Land cells: offsets of the ODYSSEY periods against CIRRUS 7/2024–17/11/2025\n"
             "grey bars: corrected years, offset fitted without them")
ax = axes[1]
mk = sorted(MON)
ax.plot([dt.datetime(y, m, 15) for y, m in mk], [MON[k]["era5"] / MON[k]["cerra"] for k in mk], ".-", color="tab:purple",
        lw=0.8, label="ERA5-Land / CERRA-Land, 06–06 UTC days")
ax.set_ylabel("ERA5-Land /\nCERRA-Land")
ax.legend(loc="upper left", fontsize=8.5)
for a in axes:
    for d in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        a.axvline(d, color="0.4", ls="--", lw=0.8)
    a.grid(alpha=0.3)
axes[1].set_xlim(axes[0].get_xlim())
fig.savefig(FIG / "level-offsets-periods.png", dpi=110)
plt.show()

## 6. Uncertainty of the level, and residual of one offset (comments of 16/11/2026 on #96)

Two tables, as asked on #96 (second comment of 16/11/2026):

1. **Uncertainty of the level**: how well the one offset of a period is known, over the area and the year. Terms:
   the 95 % interval of the offset (u = half-width / t(0.975, 11)); the bridge, as the gap between ERA5-Land and
   CERRA-Land over Europe (a choice between two alternatives: rectangular, u = D / √12, JCGM 100:2008, 4.3.7) or,
   over Belgium, the 95 % interval of the ERA5-Land / stations trend (notebook 08, section 4c) times the years from
   the middle of the period to the middle of D (an interval of a trend: u = half-width / 1.96); the target D is not
   estimated (#88). Independent terms in quadrature (5.1.2); U = 2u; the interval is written around the offset.
2. **Residual of one offset**: how far the one offset stays off for a season or a ring, with its sign and the
   number of CIRRUS years behind it. The offset is balanced over the 12 calendar months, so the seasons average
   out over a year; the rings average out over the area. These do not move the level.

In [ ]:
T11 = stats.t.ppf(0.975, 11)
TREND_LO, TREND_HI = np.log10(0.95) / 10, np.log10(1.11) / 10          # notebook 08, section 4c
MID = {p: np.mean([k[0] + (k[1] - 0.5) / 12 for k in KEYS[p]]) for p in "ABCD"}
LEVEL = {}
print("1. Uncertainty of the level (log10; u in brackets)")
print("period  offset            u offset   u bridge EU   u bridge BE   U EU (95 %)              U BE (95 %)")
for p in "ABC":
    o = OFF[p]["off"]
    u_off = (OFF[p]["hi"] - OFF[p]["lo"]) / 2 / T11
    u_eu = abs(offset_daily(p, "cerra")[0] - offset_daily(p, "era5")[0]) / np.sqrt(12)
    u_be = (TREND_HI - TREND_LO) * (MID["D"] - MID[p]) / 2 / 1.96
    U_eu, U_be = 2 * np.hypot(u_off, u_eu), 2 * np.hypot(u_off, u_be)
    LEVEL[p] = dict(u_off=u_off, u_eu=u_eu, u_be=u_be, U_eu=U_eu, U_be=U_be)
    iv = lambda U: f"×{10 ** (o - U):.2f}–{10 ** (o + U):.2f} (±{16 * U:.1f} dB)"
    print(f"{p}       ×{10 ** o:.2f} ({o:+.3f})   {u_off:.3f}      {u_eu:.3f}         {u_be:.3f}         {iv(U_eu)}   {iv(U_be)}")

D_YEARS = {s: len({k[0] + (k[1] == 12) for k in KEYS["D"] if k[1] in ms}) for s, ms in SEASONS.items()}
print("\n2. Residual of one offset (log10, layer minus the single offset)")
print("period  " + "  ".join(f"{s:>6s}" for s in SEASONS) + "   near ring  far ring   U level (EU)")
for p in "ABC":
    o = OFF[p]["off"]
    res = [OFF[p]["by_month"][[m - 1 for m in ms]].mean() - o for ms in SEASONS.values()]
    rn = offset(KEYS[p], KEYS["D"], cells=np.where(DIST < 50)[0])["off"] - o
    rf = offset(KEYS[p], KEYS["D"], cells=np.where((DIST >= 50) & (DIST < 100))[0])["off"] - o
    flag = lambda x: "*" if abs(x) > LEVEL[p]["U_eu"] else " "
    print(f"{p}       " + "  ".join(f"{x:+.3f}{flag(x)}" for x in res) + f"   {rn:+.3f}{flag(rn)}    {rf:+.3f}{flag(rf)}    {LEVEL[p]['U_eu']:.3f}")
print("CIRRUS years behind each season:", D_YEARS, "(* residual larger than U of the level)")
print("\nlocal checks against the largest level term (u offset):")
for p in "ABC":
    print(f"  {p}: Belgian bridge {LEVEL[p]['u_be'] / LEVEL[p]['u_off']:.2f}" + ("" if p != "C" else f", VMM {0.024 / np.sqrt(12) / LEVEL[p]['u_off']:.2f}"))

## 7. Volume or intensity: how often it rains and how hard

The offsets above are on the volume: the ratio of the monthly sums. A monthly sum is the number of wet hours times
their mean intensity, so per month and exactly (sums over the 186 cells, a wet cell-hour has at least 0.1 mm,
the threshold of *Radar against ERA5-Land, month by month* in `docs/method.md`):

log10(radar / ERA5-Land) = frequency + intensity + below-threshold share,

with frequency = log10 of the ratio of the wet cell-hours, intensity = log10 of the ratio of their mean rain, and
the last term the rain below the threshold. The balanced offset is linear, so the offset on the volume splits the
same way. A shift in dBZ multiplies every rain rate: it is a correction of the intensity, and it moves the
frequency only through the pixels near the threshold.

1. The split of the offsets of A, B and C, with thresholds 0.1, 0.5 and 1 mm as a check.
2. The quantiles of the wet cell-hours (median, 90th and 99th percentile) as offsets: one factor fits every
   intensity only if they agree.
3. The steps at 12/2015, 8/2018 and 7/2024 (`opera_ds.homog.step`) on frequency and intensity.
4. The volume offsets applied to the radar hours, and the split again: what they leave.

In [ ]:
THR, QS = (0.1, 0.5, 1.0), (50, 90, 99)


def wet_stats(factor=None):
    """Per month: wet cell-hour count and sum per threshold, quantiles of the wet cell-hours (0.1 mm), volume."""
    out = {}
    for y, m in MONTHS:
        product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
        ym = f"{y}-{m:02d}"
        z = np.load(OUT / (f"era5grid_odyssey_clean_{ym}.npz" if product == "odyssey" else f"era5grid_cirrus_{ym}.npz"))
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        sel = np.array([pos[c] for c in COMMON])
        he = np.array(z["hour_end"], dtype="datetime64[ns]")
        ok = (z["frames"] == FULL_FRAMES[product]) & (he <= era["valid_time"].values[-1]) & (he <= D_END)
        hours = he[ok][expver.sel(valid_time=he[ok]).values == 1]
        r = (z["rain_clean"] if product == "odyssey" else z["rain"])[ok][expver.sel(valid_time=he[ok]).values == 1][:, sel]
        if factor is not None:
            r = r * factor.get(next(p for p in "ABCD" if (y, m) in KEYS[p]), 1.0)
        e = era["tp"].sel(valid_time=hours).values[:, CI, CJ]
        good = np.isfinite(r) & np.isfinite(e)
        row = dict(vol=np.where(good, r, 0).sum() / np.where(good, e, 0).sum())
        for t in THR:
            wr, we = good & (r >= t), good & (e >= t)
            row[t] = (wr.sum(), we.sum(), r[wr].sum(), e[we].sum())
        wr, we = good & (r >= 0.1), good & (e >= 0.1)
        row["q"] = {q: (np.percentile(r[wr], q), np.percentile(e[we], q)) for q in QS}
        out[(y, m)] = row
    return out


def split(W, t=0.1):
    n_r, n_e, s_r, s_e = (np.array([W[k][t][i] for k in MONTHS]) for i in range(4))
    vol = np.log10([W[k]["vol"] for k in MONTHS])
    freq, inten = np.log10(n_r / n_e), np.log10(s_r / n_r / (s_e / n_e))
    return {"volume": vol, "frequency": freq, "intensity": inten, "below threshold": vol - freq - inten}


def balanced(series, p):
    """Offset of a monthly log10 series, period p against D, balanced over the 12 calendar months."""
    pos = {k: i for i, k in enumerate(MONTHS)}
    s = lambda q: np.array([series[pos[k]] for k in KEYS[q]])
    mp, md = np.array([k[1] for k in KEYS[p]]), np.array([k[1] for k in KEYS["D"]])
    d = np.array([s("D")[md == m].mean() - s(p)[mp == m].mean() for m in range(1, 13)])
    return d.mean(), stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12), d


W = wet_stats()
for t in THR:
    print(f"threshold {t} mm per cell-hour (log10; ×factor [95%])")
    sp = split(W, t)
    for p in "ABC":
        print(f"  {p}: " + "   ".join(f"{name} {o:+.3f} ×{10 ** o:.2f} [×{10 ** (o - h):.2f}–{10 ** (o + h):.2f}]"
                                     for name, (o, h, _) in ((n, balanced(v, p)) for n, v in sp.items())))
print("\nquantiles of the wet cell-hours, radar / ERA5-Land, as offsets (log10)")
for q in QS:
    s = np.log10([W[k]["q"][q][0] / W[k]["q"][q][1] for k in MONTHS])
    print(f"  q{q}: " + "   ".join(f"{p} {balanced(s, p)[0]:+.3f} ±{balanced(s, p)[1]:.3f}" for p in "ABC"))
sp = split(W)
print("\nseasons (log10, period against D)")
for name in ("frequency", "intensity"):
    for p in "ABC":
        d = balanced(sp[name], p)[2]
        print(f"  {name:9s} {p}: " + "  ".join(f"{s} {d[[m - 1 for m in ms]].mean():+.3f}" for s, ms in SEASONS.items()))

In [ ]:
print("steps (log10, after minus before; t-test p); 7/2024 without 5/2020")
for at in ((2015, 12), (2018, 8), (2024, 7)):
    for n in (12, 16):
        cols = []
        for name in ("volume", "frequency", "intensity"):
            r = step(dict(zip(MONTHS, 10 ** sp[name])), at, n, exclude=[(2020, 5)] if at == (2024, 7) else ())
            cols.append(f"{name} {r['mean_after'] - r['mean_before']:+.3f} (p {r['t_p']:.3f})")
        print(f"  {at[1]}/{at[0]}, {n} months each side: " + "   ".join(cols))

FAC = {p: 10 ** OFF[p]["off"] for p in "ABC"}
Wc = wet_stats(FAC)
spc = split(Wc)
print("\nafter the volume offsets (log10, D minus the corrected period; negative: the period is now above D)")
for p in "ABC":
    qs = [balanced(np.log10([Wc[k]["q"][q][0] / Wc[k]["q"][q][1] for k in MONTHS]), p)[0] for q in QS]
    print(f"  {p}: " + "   ".join(f"{n} {balanced(spc[n], p)[0]:+.3f}" for n in ("volume", "frequency", "intensity"))
          + "   q50/q90/q99 " + " ".join(f"{x:+.3f}" for x in qs))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True, constrained_layout=True)
t = [dt.datetime(y, m, 15) for y, m in MONTHS]
for ax, name, lab in ((axes[0], "frequency", "wet cell-hours,\nradar / ERA5-Land (log10)"),
                      (axes[1], "intensity", "mean of the wet cell-hours,\nradar / ERA5-Land (log10)")):
    ax.plot(t, sp[name], color="0.55", lw=0.9, marker=".", ms=3)
    for p, col in zip("ABCD", ("tab:orange", "tab:green", "tab:blue", "tab:red")):
        i = [list(MONTHS).index(k) for k in KEYS[p]]
        ax.hlines(np.mean(sp[name][i]), t[i[0]], t[i[-1]], color=col, lw=2.2, label=p)
    for d in (dt.datetime(2015, 12, 1), dt.datetime(2018, 8, 1), dt.datetime(2024, 7, 1)):
        ax.axvline(d, color="0.4", ls="--", lw=0.8)
    ax.set_ylabel(lab)
    ax.grid(alpha=0.3)
axes[0].legend(loc="upper left", ncol=4, fontsize=8.5, title="period mean", title_fontsize=8.5)
axes[0].set_title("Over Belgium, 186 ERA5-Land cells; a wet cell-hour has at least 0.1 mm", fontsize=10)
fig.savefig(FIG / "level-frequency-intensity.png", dpi=110)
plt.show()

## 8. Level on the intensity, and the frequency gap documented (comments of 17/11/2026 on #96)

Proposal for G1 on #96: correct the intensity only, as a shift in dBZ, and document the frequency gap. The extra
echoes of CIRRUS against C sit in hours that ERA5-Land and the RMI stations call dry (`scripts/qc_dry_wet_hours.py`),
so matching CIRRUS in frequency would add echoes that are not rain.

- **Offset term:** the intensity offsets of section 7 (0.1 mm per cell-hour), t interval with 11 df, u = half / 2.20.
- **Bridge over Europe:** the same split is not possible on CERRA-Land, which is daily. The intensity offset is
  computed on the 06–06 days of section 3 (area mean of the 186 cells) against ERA5-Land and against CERRA-Land,
  with a wet day of at least 1 mm (the daily wet threshold of Goudenhoofdt and Delobbe 2009); the gap D between
  the two gives u = D/√12, as in section 6.
- **Bridge over Belgium:** the trend of the wet-day intensity of ERA5-Land against the 12 RMI stations of notebook
  08, section 4c (same days, months and lag-1 correction; wet day of at least 1 mm on both), times the years
  between the middle of the period and the middle of D, u = half-width / 1.96.
- **Residual of one offset:** seasons from the monthly differences; rings 0–50 and 50–100 km from the intensity
  offset on the cells of each ring.
- **Frequency gap:** the frequency offsets of section 7, and their split into hours that ERA5-Land calls dry and
  wet (`results/h3_level/qc_dry_wet_era5.csv`).

In [ ]:
import pandas as pd
import importlib.util
from scipy.stats import linregress

# --- 8a. intensity offsets of A, B, C against D (section 7, 0.1 mm), with the offset term
SP = split(W)
IOFF = {}
for p in "ABC":
    o, h, d = balanced(SP["intensity"], p)
    IOFF[p] = dict(off=o, half=h, by_month=d, u_off=h / T11)

# --- 8b. bridge over Europe: the same intensity offset on 06-06 days, against ERA5-Land and against CERRA-Land.
# Area means over the 186 cells (as offset_daily); a wet day has at least 1 mm (Goudenhoofdt and Delobbe 2009).
WET_DAY = 1.0


def inten_daily(p, ref, thr=WET_DAY):
    lr = {}
    for m_ in np.unique(ym):
        k = ym == m_
        if k.sum() < 15:
            continue
        r_, x_ = rd[k], ref[k]
        if (r_ >= thr).sum() < 2 or (x_ >= thr).sum() < 2:
            continue
        lr[(int(str(m_)[:4]), int(str(m_)[5:7]))] = np.log10(r_[r_ >= thr].mean() / x_[x_ >= thr].mean())
    a, b = PERIODS[p]
    d = np.array([np.mean([v for k, v in lr.items() if k[1] == mo and PERIODS["D"][0] <= k <= PERIODS["D"][1]]) -
                  np.mean([v for k, v in lr.items() if k[1] == mo and a <= k <= b]) for mo in range(1, 13)])
    return d.mean(), stats.t.ppf(0.975, 11) * d.std(ddof=1) / np.sqrt(12)


for p in "ABC":
    e_, c_ = inten_daily(p, ed)[0], inten_daily(p, cd)[0]
    IOFF[p].update(daily_era5=e_, daily_cerra=c_, u_eu=abs(c_ - e_) / np.sqrt(12),
                   daily_era5_01=inten_daily(p, ed, 0.1)[0], daily_cerra_01=inten_daily(p, cd, 0.1)[0])

# --- 8c. bridge over Belgium: trend of the wet-day intensity of ERA5-Land against the RMI stations
# (notebook 08, section 4c: fixed set of 12 stations, validated hours, complete 06-06 days, monthly pools with
# every station present, calendar-month means removed, lag-1 corrected interval); wet day >= 1 mm on both.
aws = np.load(DATA_DIR / "rmi_aws" / "aws_1hour_precip.npz")
ST = {s["code"]: s for s in json.load(open(DATA_DIR / "rmi_aws" / "stations.json", encoding="utf-8"))}
DAYS = np.arange(np.datetime64("2012-01-02T06", "h"), np.datetime64("2025-11-30T07", "h"), np.timedelta64(24, "h"))


def to_days(t, v):
    lab_ = (t - np.timedelta64(7, "h")).astype("datetime64[D]").astype("datetime64[h]") + np.timedelta64(30, "h")
    i = np.searchsorted(DAYS, lab_)
    m_ = (i < DAYS.size) & (DAYS[np.minimum(i, DAYS.size - 1)] == lab_)
    tot, n = np.zeros(DAYS.size), np.zeros(DAYS.size)
    np.add.at(tot, i[m_], np.nan_to_num(v[m_]))
    np.add.at(n, i[m_], np.isfinite(v[m_]))
    return np.where(n == 24, tot, np.nan)


SD, ED = {}, {}
for code in ST:
    k = aws["code"] == code
    SD[code] = to_days(aws["time"][k].astype("datetime64[h]"), np.where(aws["validated"][k], aws["precip"][k], np.nan))
dyr = DAYS.astype("datetime64[Y]").astype(int) + 1970
CODES = [x for x in SD if all(np.isfinite(SD[x][dyr == y]).mean() >= 0.8 for y in range(2012, 2026))]
eh = era["valid_time"].values.astype("datetime64[h]")
for code in CODES:
    i = int(np.abs(era.latitude.values - ST[code]["lat"]).argmin())
    j = int(np.abs(era.longitude.values - ST[code]["lon"]).argmin())
    ED[code] = to_days(eh, era["tp"].isel(latitude=i, longitude=j).values)
DM = (DAYS - np.timedelta64(30, "h")).astype("datetime64[M]")
UM = np.unique(DM)
xm = np.array([int(str(m_)[:4]) + (int(str(m_)[5:7]) - 0.5) / 12 for m_ in UM])
mm = np.array([int(str(m_)[5:7]) for m_ in UM])


def trend(lr):
    an = lr - np.array([np.nanmean(lr[mm == q]) for q in mm])
    k = np.isfinite(an)
    r = linregress(xm[k], an[k])
    res = an[k] - (r.intercept + r.slope * xm[k])
    r1 = np.corrcoef(res[:-1], res[1:])[0, 1]
    neff = k.sum() * (1 - r1) / (1 + r1)
    se = r.stderr * np.sqrt((k.sum() - 2) / (neff - 2))
    half = stats.t.ppf(0.975, neff - 2) * se
    return r.slope, half


ns, ss, ne, se_ = (np.full((len(CODES), UM.size), np.nan) for _ in range(4))
for a, code in enumerate(CODES):
    ok = np.isfinite(SD[code]) & np.isfinite(ED[code])
    for b, m_ in enumerate(UM):
        k = ok & (DM == m_)
        if k.sum() >= 20:
            s_, e_ = SD[code][k], ED[code][k]
            ns[a, b], ss[a, b] = (s_ >= WET_DAY).sum(), s_[s_ >= WET_DAY].sum()
            ne[a, b], se_[a, b] = (e_ >= WET_DAY).sum(), e_[e_ >= WET_DAY].sum()
full = np.isfinite(ns).all(0)
pool = lambda X: np.where(full, np.nansum(X, 0), np.nan)
inten_ratio = np.log10((pool(se_) / pool(ne)) / (pool(ss) / pool(ns)))
vol_ratio = np.log10(pool(se_) / pool(ss))
slope_i, half_i = trend(inten_ratio)
slope_v, half_v = trend(vol_ratio)
print(f"RMI, {len(CODES)} stations, {full.sum()} months; ERA5-Land / stations a decade:"
      f" wet-day intensity ×{10 ** (10 * slope_i):.2f} (95% ×{10 ** (10 * (slope_i - half_i)):.2f}–×{10 ** (10 * (slope_i + half_i)):.2f}),"
      f" totals of the wet days ×{10 ** (10 * slope_v):.2f} (×{10 ** (10 * (slope_v - half_v)):.2f}–×{10 ** (10 * (slope_v + half_v)):.2f})")
TI_LO, TI_HI = slope_i - half_i, slope_i + half_i
for p in "ABC":
    IOFF[p]["u_be"] = (TI_HI - TI_LO) * (MID["D"] - MID[p]) / 2 / 1.96

# --- 8d. the table
print("\n1. Uncertainty of the level on the intensity (log10; u)")
print("period  offset            u offset  u bridge EU  u bridge BE  U EU (95%)                 U BE (95%)")
for p in "ABC":
    o, L = IOFF[p]["off"], IOFF[p]
    L["U_eu"], L["U_be"] = 2 * np.hypot(L["u_off"], L["u_eu"]), 2 * np.hypot(L["u_off"], L["u_be"])
    iv = lambda U: f"×{10 ** (o - U):.2f}–{10 ** (o + U):.2f} (±{16 * U:.1f} dB)"
    print(f"{p}       ×{10 ** o:.2f} ({o:+.3f})   {L['u_off']:.3f}     {L['u_eu']:.3f}        {L['u_be']:.3f}"
          f"        {iv(L['U_eu'])}   {iv(L['U_be'])}")
print("daily check (06-06, area mean, wet day >= 1 mm; >= 0.1 mm in brackets): ERA5-Land / CERRA-Land offsets")
for p in "ABC":
    L = IOFF[p]
    print(f"   {p}: {L['daily_era5']:+.3f} / {L['daily_cerra']:+.3f}   ({L['daily_era5_01']:+.3f} / {L['daily_cerra_01']:+.3f})")

# --- 8e. residual of one offset: seasons and rings
RINGS_I = {"0-50 km": np.where(DIST < 50)[0], "50-100 km": np.where((DIST >= 50) & (DIST < 100))[0]}


def wet_ring(cells):
    out = {}
    for y, m in MONTHS:
        product = "odyssey" if (y, m) < (2024, 7) else "cirrus"
        ym_ = f"{y}-{m:02d}"
        z = np.load(OUT / (f"era5grid_odyssey_clean_{ym_}.npz" if product == "odyssey" else f"era5grid_cirrus_{ym_}.npz"))
        pos = {tuple(c): k for k, c in enumerate(z["cells"])}
        sel = np.array([pos[COMMON[c]] for c in cells])
        he = np.array(z["hour_end"], dtype="datetime64[ns]")
        ok = (z["frames"] == FULL_FRAMES[product]) & (he <= era["valid_time"].values[-1]) & (he <= D_END)
        fin = expver.sel(valid_time=he[ok]).values == 1
        r = (z["rain_clean"] if product == "odyssey" else z["rain"])[ok][fin][:, sel]
        e = era["tp"].sel(valid_time=he[ok][fin]).values[:, CI[cells], CJ[cells]]
        good = np.isfinite(r) & np.isfinite(e)
        wr, we = good & (r >= 0.1), good & (e >= 0.1)
        out[(y, m)] = np.log10(r[wr].mean() / e[we].mean())
    return np.array([out[k] for k in MONTHS])


print("\n2. Residual of one offset on the intensity (log10; * above U EU of the level)")
RES_I = {}
ring_series = {name: wet_ring(cells) for name, cells in RINGS_I.items()}
for p in "ABC":
    o, U = IOFF[p]["off"], IOFF[p]["U_eu"]
    res = [IOFF[p]["by_month"][[m - 1 for m in ms]].mean() - o for ms in SEASONS.values()]
    res += [balanced(ring_series[n], p)[0] - o for n in RINGS_I]
    RES_I[p] = res
    print(f"   {p}: " + "  ".join(f"{n} {x:+.3f}{'*' if abs(x) > U else ' '}" for n, x in zip(list(SEASONS) + list(RINGS_I), res))
          + f"   U {U:.3f}")

# --- 8f. the frequency gap per period, documented, with its dry-hour split (scripts/qc_dry_wet_hours.py)
_spec = importlib.util.spec_from_file_location("qc", Path.cwd().parent / "scripts" / "qc_dry_wet_hours.py")
qc = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(qc)
QE = pd.read_csv(OUT / "qc_dry_wet_era5.csv")
for p_ in qc.PERIODS:
    qc.KEYS[p_] = sorted({(y, m) for y, m, q in zip(QE.year, QE.month, QE.period) if q == p_})
print("\n3. Frequency gap per period against D (log10, 95%), and where it sits (absolute F = FA + H, ERA5-Land)")
for p in "ABC":
    o, h, _ = balanced(SP["frequency"], p)
    g = qc.balanced_gap(QE, p, "D")
    print(f"   {p}: ×{10 ** o:.2f} ({o:+.3f} ±{h:.3f}); dF {g['F'][0]:+.3f}: dry hours {g['FA'][0]:+.3f} ±{g['FA'][1]:.3f},"
          f" wet hours {g['H'][0]:+.3f} ±{g['H'][1]:.3f}; share in dry hours {g['FA'][0] / g['F'][0]:.2f}")


## 9. Continuity and jumps with no date, on the intensity (comment of 18/11/2026 on #96)

Two of the three checks asked on #96 before G1, on the corrected wet-hour intensity radar / ERA5-Land of section 7
(0.1 mm per cell-hour), corrected with the intensity offsets of section 8 (A ×1.34, B ×1.06, C ×1.01).

- **Continuity at the period boundaries:** the test of section 4, recomputed on the intensity. Same 12-month blocks,
  each left out in turn and corrected with the intensity offset refitted on the rest; yardstick and jumps built the
  same way. The jumps are also given with the offsets of the whole period.
- **Jumps with no date inside A, B and C** (acceptance criterion of #20): SNHT (Alexandersson 1986), the maximal t
  test that WMO (2020, sections 2.3.1 and 3.2.1) names for one break at an unknown date. It runs on the corrected
  monthly intensity of each period, with the calendar-month means of the period removed (as notebook 08) and the
  artefact months (3–6/2018, 5/2020) left out. Consecutive months are autocorrelated (WMO 2020, section 2.2.2), so the
  95 % critical value and the p value come from Monte Carlo series (WMO 2020, section 5.3.2), AR(1) with the lag-1
  correlation of the series, with the same calendar-month means removed. Pettitt (1979; Ribeiro et al. 2016, section
  3.1) is run the same way as a check. A jump found this way is a candidate, not a new period.

The correction is applied here as a constant on the log ratio. On the pixels it is a shift in dBZ with the 7 dBZ
threshold and the 55 dBZ cap, which moves part of the shift into the frequency (closure test on #100).

In [ ]:
# --- 9a. continuity on the intensity (comment of 18/11/2026 on #96): section 4, on the wet-hour intensity
# radar / ERA5-Land (section 7, 0.1 mm per cell-hour) instead of the volume. Same 12-month blocks aligned to the
# end of each period, each left out in turn and corrected with the intensity offset refitted on the rest; the
# yardstick is the standard deviation of the corrected means of the full blocks of A, B and C (95 %, chi-square).
SI = split(W)["intensity"]
IPOS = {k: i for i, k in enumerate(MONTHS)}


def ival(keys):
    return np.array([SI[IPOS[k]] for k in keys])


def ioffset(c_keys):
    lc, ld = ival(c_keys), ival(KEYS["D"])
    mc, md = np.array([k[1] for k in c_keys]), np.array([k[1] for k in KEYS["D"]])
    return np.mean([ld[md == m].mean() - lc[mc == m].mean() for m in range(1, 13)])


V1 = {p: ioffset(KEYS[p]) for p in "ABC"}                       # ×1.34, ×1.06, ×1.01 (section 8)


def icorrected(p, blk, exclude=()):
    o = ioffset([k for k in KEYS[p] if k not in blk and k not in exclude])
    blk = [k for k in blk if k not in exclude]
    return (ival(blk) + o).mean(), o


print("9a. Continuity on the intensity (log10)")
print("   intensity offsets: " + ", ".join(f"{p} {V1[p]:+.3f} (×{10 ** V1[p]:.2f})" for p in "ABC"))
IROWS = []
for p in "ABC":
    for blk in blocks(p):
        m, o = icorrected(p, blk)
        IROWS.append((p, blk, m))
        print(f"   {p} {blk[0][1]}/{blk[0][0]}–{blk[-1][1]}/{blk[-1][0]} ({len(blk)} months): offset of the rest {o:+.3f},"
              f" corrected mean {m:+.3f}")
ifull = np.array([m for p, blk, m in IROWS if len(blk) == 12])
n_i, IYARD = ifull.size, ifull.std(ddof=1)
IY_LO, IY_HI = (IYARD * np.sqrt((n_i - 1) / stats.chi2.ppf(q, n_i - 1)) for q in (0.975, 0.025))
print(f"   yardstick: {IYARD:.3f}, 95% {IY_LO:.3f}–{IY_HI:.3f} ({n_i} blocks)")
ID12 = ival([k for k in KEYS["D"] if k <= (2025, 6)]).mean()
IJUMPS = {}
for p, q in (("A", "B"), ("B", "C"), ("C", "D")):
    before = icorrected(p, KEYS[p][-12:])[0]
    after = ID12 if q == "D" else icorrected(q, KEYS[q][:12])[0]
    fixed = (ID12 if q == "D" else ival(KEYS[q][:12]).mean() + V1[q]) - (ival(KEYS[p][-12:]).mean() + V1[p])
    IJUMPS[(p, q)] = after - before
    print(f"   jump {p}→{q}: {after - before:+.3f} (×{10 ** (after - before):.2f}) → "
          f"{'within' if abs(after - before) <= IYARD else 'larger than'} the yardstick;"
          f" with the offsets of all the period {fixed:+.3f}")
ib8 = icorrected("B", KEYS["B"][-12:], exclude=spring18)[0]
print(f"   jump B→C without 3–6/2018 in the fit and in the block: {icorrected('C', KEYS['C'][:12])[0] - ib8:+.3f}")
print(f"   for comparison, section 4 on the volume: yardstick {YARD:.3f}; jumps "
      + ", ".join(f"{p}→{q} {b - a:+.3f}" for (p, q), (a, b) in JUMPS.items()))

# --- 9b. jumps with no date inside A, B and C (acceptance criterion of #20). SNHT (Alexandersson 1986), the
# maximal t test that WMO (2020, section 2.3.1 and 3.2.1) names for one break at an unknown date, on the corrected
# monthly intensity of each period, calendar-month means of the period removed (as notebook 08), artefact months out.
# Consecutive months are autocorrelated (WMO 2020, section 2.2.2): the 95 % critical value and the p value come from
# Monte Carlo series (WMO 2020, section 5.3.2) with the lag-1 correlation of the series and the same calendar months
# removed. Pettitt (1979; rank test, Ribeiro et al. 2016, section 3.1) is run the same way as a check.
ARTEFACTS = set(spring18) | {(2020, 5)}
RNG = np.random.default_rng(20261119)
N_MC = 20000


def deseason(x, mo):
    return x - np.array([x[mo == m].mean() for m in mo])


def snht(x):
    z = (x - x.mean()) / x.std(ddof=1)
    n, c = z.size, np.cumsum(z)
    k = np.arange(1, n)
    t = k * (c[:-1] / k) ** 2 + (n - k) * ((c[-1] - c[:-1]) / (n - k)) ** 2
    return t.max(), int(t.argmax()) + 1                     # statistic, size of the first segment


def pettitt(x):
    s = np.sign(x[None, :] - x[:, None])
    u = np.array([s[:t + 1, t + 1:].sum() for t in range(x.size - 1)])
    return np.abs(u).max(), int(np.abs(u).argmax()) + 1


def mc(stat, n, mo, r1):
    out = np.empty(N_MC)
    for i in range(N_MC):
        e = RNG.standard_normal(n)
        x = np.empty(n)
        x[0] = e[0]
        for j in range(1, n):
            x[j] = r1 * x[j - 1] + np.sqrt(1 - r1 ** 2) * e[j]
        out[i] = stat(deseason(x, mo))[0]
    return out


print("\n9b. Jumps with no date inside each period (corrected monthly intensity, calendar-month means removed)")
BREAKS = {}
for p in "ABC":
    keys = [k for k in KEYS[p] if k not in ARTEFACTS]
    mo = np.array([k[1] for k in keys])
    x = deseason(ival(keys) + V1[p], mo)
    r1 = max(np.corrcoef(x[:-1], x[1:])[0, 1], 0.0)
    for name, f in (("SNHT", snht), ("Pettitt", pettitt)):
        s, k = f(x)
        null = mc(f, x.size, mo, r1)
        crit, pval = np.quantile(null, 0.95), (null >= s).mean()
        size = x[k:].mean() - x[:k].mean()
        BREAKS[(p, name)] = dict(date=keys[k], stat=s, crit=crit, p=pval, size=size, n=x.size, r1=r1)
        print(f"   {p} ({x.size} months, lag-1 r {r1:.2f}) {name}: first month after the most likely break "
              f"{keys[k][1]}/{keys[k][0]}, statistic {s:.1f} (95% {crit:.1f}), p {pval:.3f};"
              f" step {size:+.3f} (×{10 ** size:.2f}) → {'candidate' if pval < 0.05 else 'not significant'}")


## Findings

See `docs/method.md`, *Level offsets of periods A and B (second H3 pull request)*, *Uncertainty budget of the level
correction (for G1)*, *Volume or intensity: how often it rains and how hard* and *Level on the intensity (proposal of
17/11/2026 for G1)* and *Continuity and jumps on the intensity (for G1)*.